# 3. Train Masking Model and Count Decoder

Fine-tune PerturbGen on the LPS tokenized data from notebook 02.

**Requirements**
- CUDA GPU (this step will not train on CPU)
- Poetry / `.venv` kernel with `perturbgen` installed
- `geneformer` installed (`pip install git+https://huggingface.co/ctheodoris/Geneformer@0960cf63969aa0dfdb00c6fd46316a3fbe7a1c9b`)
- Enough disk for checkpoints (several GB)

## 3.0. Shared paths and environment

`perturbgen` changes the working directory to the workspace root (`/home/stuke1/perturbgen`), where `T_perturb/` lives. Paths below match that layout.

Training is restricted to the **last three GPUs** (`CUDA_VISIBLE_DEVICES=5,6,7`). Check `nvidia-smi` first — those cards must have free memory.

In [2]:
import os
from pathlib import Path

# Workspace root (== perturbgen.configs.paths.ROOT)
WORKSPACE = Path("/home/stuke1/perturbgen")
REPO = WORKSPACE / "Perturbgen"
TOKENIZED = WORKSPACE / "T_perturb" / "tokenized_data" / "LPS_all_tps_2k"

# Tokenized LPS outputs from notebook 02
SRC_DATASET = str(TOKENIZED / "dataset_2000_hvg_src" / "normal.dataset")
TGT_DATASET_FOLDER = str(TOKENIZED / "dataset_2000_hvg_tgt")
SRC_ADATA = str(TOKENIZED / "h5ad_pairing_2000_hvg_src" / "normal.h5ad")
TGT_ADATA_FOLDER = str(TOKENIZED / "h5ad_pairing_2000_hvg_tgt")
MAPPING_DICT_PATH = str(TOKENIZED / "token_id_to_genename_2000_hvg.pkl")

# Pretrained encoder shipped with the repo
ENCODER_PATH = str(
    REPO
    / "pretraining_cohort"
    / "20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_42-epoch=00.ckpt"
)

VAR_LIST = ["cell_type_harmonized", "time_after_LPS"]
PRED_TPS = ["1", "2", "3"]  # 1=90m, 2=6h, 3=10h

# Use only the last three physical GPUs (seen as cuda:0,1,2 inside the job)
os.environ["CUDA_VISIBLE_DEVICES"] = "5,6,7"
os.environ.setdefault("WANDB_MODE", "offline")

required = [
    SRC_DATASET,
    TGT_DATASET_FOLDER,
    SRC_ADATA,
    TGT_ADATA_FOLDER,
    MAPPING_DICT_PATH,
    ENCODER_PATH,
]
missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing paths:\n" + "\n".join(missing))

print("WORKSPACE:", WORKSPACE)
print("TOKENIZED:", TOKENIZED)
print("ENCODER:", ENCODER_PATH)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("All required paths exist.")

WORKSPACE: /home/stuke1/perturbgen
TOKENIZED: /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k
ENCODER: /home/stuke1/perturbgen/Perturbgen/pretraining_cohort/20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_42-epoch=00.ckpt
CUDA_VISIBLE_DEVICES: 5,6,7
All required paths exist.


## 3.1. Train masking model (GPU required)

Uses GPUs **5,6,7** (set in section 3.0). With free A100 40 GB cards, `BATCH_SIZE = 64` is the paper default. If you hit OOM (e.g. other jobs on those GPUs), lower to `16` or `8`.

In [15]:
MASK_OUTPUT_DIR = str(WORKSPACE / "T_perturb" / "res" / "masking")

BATCH_SIZE = 64      # A100 40GB default; drop to 16/8 if OOM
EPOCHS = 20
CELLGEN_LR = 1e-4
CELLGEN_WD = 1e-4
N_WORKERS = 4
NUM_LAYERS = 6
D_FF = 32
D_MODEL = 768
SEED = 0
NUM_NODE = 1         # single node; 3 GPUs via CUDA_VISIBLE_DEVICES

CONTEXT_MODE = "True"
POS_ENCODING_MODE = "time_pos_sin"
MASK_SCHEDULER = "pow"
USE_WEIGHTED_SAMPLER = "False"

# Optional: set to a real .ckpt path to resume; leave None for a fresh run
CKPT_MASKING_RESUME = None

In [16]:
mask_cmd = [
    "python",
    "-m",
    "perturbgen",
    "train-mask",
    "--train_mode", "masking",
    "--split", "False",
    "--encoder", "scmaskgit",
    "--splitting_mode", "stratified",
    "--split_obs", "cell_type_harmonized",
    "--output_dir", MASK_OUTPUT_DIR,
    "--src_dataset", SRC_DATASET,
    "--tgt_dataset_folder", TGT_DATASET_FOLDER,
    "--src_adata", SRC_ADATA,
    "--tgt_adata_folder", TGT_ADATA_FOLDER,
    "--mapping_dict_path", MAPPING_DICT_PATH,
    "--batch_size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--cellgen_lr", str(CELLGEN_LR),
    "--cellgen_wd", str(CELLGEN_WD),
    "--n_workers", str(N_WORKERS),
    "--num_layers", str(NUM_LAYERS),
    "--d_ff", str(D_FF),
    "--pred_tps", *PRED_TPS,
    "--var_list", *VAR_LIST,
    "--encoder_path", ENCODER_PATH,
    "--seed", str(SEED),
    "--context_mode", CONTEXT_MODE,
    "--pos_encoding_mode", POS_ENCODING_MODE,
    "--mask_scheduler", MASK_SCHEDULER,
    "--num_node", str(NUM_NODE),
    "--d_model", str(D_MODEL),
    "--use_weighted_sampler", USE_WEIGHTED_SAMPLER,
    "--wandb_mode", "offline",
]

if CKPT_MASKING_RESUME:
    mask_cmd += ["--ckpt_masking_path", str(CKPT_MASKING_RESUME)]

print(" ".join(mask_cmd))

python -m perturbgen train-mask --train_mode masking --split False --encoder scmaskgit --splitting_mode stratified --split_obs cell_type_harmonized --output_dir /home/stuke1/perturbgen/T_perturb/res/masking --src_dataset /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset --tgt_dataset_folder /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt --src_adata /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad --tgt_adata_folder /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt --mapping_dict_path /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl --batch_size 64 --epochs 20 --cellgen_lr 0.0001 --cellgen_wd 0.0001 --n_workers 4 --num_layers 6 --d_ff 32 --pred_tps 1 2 3 --var_list cell_type_harmonized time_after_LPS --encoder_path /home/stuke1/perturbgen/Perturbgen/pretraining

In [20]:
import os
import subprocess

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "5,6,7"
env.setdefault("WANDB_MODE", "offline")

print("Launching masking train on GPUs", env["CUDA_VISIBLE_DEVICES"])
subprocess.run(mask_cmd, check=True, cwd=str(WORKSPACE), env=env)

Launching masking train on GPUs 5,6,7
loading, please wait...
Current working directory: /home/stuke1/perturbgen
Loading and preprocessing data...
Loading 3_10h_LPS.dataset...
Loading 2_6h_LPS.dataset...
Loading 1_90m_LPS.dataset...
Loading 3_10h_LPS.h5ad...


Seed set to 42
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading 2_6h_LPS.h5ad...


/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading 1_90m_LPS.h5ad...


...

Epoch 1:   0%|          | 0/2351 [00:00<?, ?it/s, v_num=gbno, lr=0.0001, train/perplexity_step=146.0, train/masking_loss_step=4.990, train/perplexity_epoch=235.0, train/masking_loss_epoch=5.380]           

/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:431: It is recommended to use `self.log('train/perplexity', ..., sync_dist=True)` when logging on epoch level in distributed setting to accumulate the metric across devices.
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:431: It is recommended to use `self.log('train/masking_loss', ..., sync_dist=True)` when logging on epoch level in distributed setting to accumulate the metric across devices.


Epoch 2:  80%|███████▉  | 1880/2351 [09:47<02:27,  3.20it/s, v_num=gbno, lr=0.0001, train/perplexity_step=118.0, train/masking_loss_step=4.770, train/perplexity_epoch=142.0, train/masking_loss_epoch=4.940]


Aborted!

Detected KeyboardInterrupt, attempting graceful shutdown ...


KeyboardInterrupt: 

## 3.2. Train count decoder (GPU required)

Uses the **best masking checkpoint** from section 3.1. Update `CKPT_MASKING_PATH` below after masking finishes (look under `T_perturb/res/masking/`).

In [6]:
from pathlib import Path

COUNT_OUTPUT_DIR = str(WORKSPACE / "T_perturb" / "res" / "count")

# Final masking checkpoint from section 3.1 (epoch 19 of 20)
CKPT_MASKING_PATH = (
    "/home/stuke1/perturbgen/T_perturb/res/masking/checkpoints/"
    "20260729_1751_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_"
    "ptime_pos_sin_m_pow_tp_1-2-3_s_0-epoch=19.ckpt"
)

BATCH_SIZE = 16      # count decoder default from tutorial; lower if OOM
EPOCHS = 16
COUNT_LR = 0.001
COUNT_WD = 0.001
CELLGEN_LR = 1e-4
CELLGEN_WD = 1e-4
MLM_PROB = 0.30
N_WORKERS = 4
NUM_LAYERS = 6
D_FF = 32
D_MODEL = 768
NUM_NODE = 1         # GPUs 5,6,7 via CUDA_VISIBLE_DEVICES
LOSS_MODE = "zinb"  # mse | nb | zinb

COUNT_DROPOUT = 0.1
USE_POSITIONAL_ENCODING = "False"
LAYER_NORM = "True"
CONTEXT_MODE = "True"
POS_ENCODING_MODE = "time_pos_sin"
MASK_SCHEDULER = "pow"

assert Path(CKPT_MASKING_PATH).exists(), f"Missing checkpoint: {CKPT_MASKING_PATH}"
print("CKPT_MASKING_PATH:", CKPT_MASKING_PATH)

CKPT_MASKING_PATH: /home/stuke1/perturbgen/T_perturb/res/masking/checkpoints/20260729_1751_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_0-epoch=19.ckpt


In [4]:
if not CKPT_MASKING_PATH:
    raise ValueError(
        "Set CKPT_MASKING_PATH to your trained masking checkpoint before running the count decoder."
    )

count_cmd = [
    "python",
    "-m",
    "perturbgen",
    "train-decoder",
    "--train_mode", "count",
    "--split", "False",
    "--splitting_mode", "stratified",
    "--output_dir", COUNT_OUTPUT_DIR,
    "--ckpt_masking_path", str(CKPT_MASKING_PATH),
    "--src_dataset", SRC_DATASET,
    "--tgt_dataset_folder", TGT_DATASET_FOLDER,
    "--src_adata", SRC_ADATA,
    "--tgt_adata_folder", TGT_ADATA_FOLDER,
    "--mapping_dict_path", MAPPING_DICT_PATH,
    "--batch_size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--count_lr", str(COUNT_LR),
    "--cellgen_lr", str(CELLGEN_LR),
    "--cellgen_wd", str(CELLGEN_WD),
    "--count_wd", str(COUNT_WD),
    "--mlm_prob", str(MLM_PROB),
    "--n_workers", str(N_WORKERS),
    "--num_layers", str(NUM_LAYERS),
    "--d_ff", str(D_FF),
    "--loss_mode", LOSS_MODE,
    "--pred_tps", *PRED_TPS,
    "--var_list", *VAR_LIST,
    "--encoder", "scmaskgit",
    "--count_dropout", str(COUNT_DROPOUT),
    "--use_positional_encoding", USE_POSITIONAL_ENCODING,
    "--layer_norm", LAYER_NORM,
    "--context_mode", CONTEXT_MODE,
    "--encoder_path", ENCODER_PATH,
    "--pos_encoding_mode", POS_ENCODING_MODE,
    "--mask_scheduler", MASK_SCHEDULER,
    "--num_node", str(NUM_NODE),
    "--d_model", str(D_MODEL),
    "--ckpt_every_n_epochs", "5",
    "--wandb_mode", "offline",
]

print(" ".join(count_cmd))

python -m perturbgen train-decoder --train_mode count --split False --splitting_mode stratified --output_dir /home/stuke1/perturbgen/T_perturb/res/count --ckpt_masking_path /home/stuke1/perturbgen/T_perturb/res/masking/checkpoints/20260729_1751_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_0-epoch=19.ckpt --src_dataset /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset --tgt_dataset_folder /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt --src_adata /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad --tgt_adata_folder /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt --mapping_dict_path /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl --batch_size 16 --epochs 16 --count_lr 0.001 --cellgen_lr 0.0001 --cellgen_wd 0.0001 --count_wd 

---
Next: gene embedding extraction and downstream analyses in notebooks 04–06.

In [ ]:
import os
import subprocess

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "5,6,7"Launching count-decoder train on GPUs 5,6,7
loading, please wait...
Current working directory: /home/stuke1/perturbgen
Loading and preprocessing data...
Loading 3_10h_LPS.dataset...
Loading 2_6h_LPS.dataset...
Loading 1_90m_LPS.dataset...
Loading 3_10h_LPS.h5ad...
Seed set to 42
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Loading 2_6h_LPS.h5ad...
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Loading 1_90m_LPS.h5ad...
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
---PerturbGen training --- 
Target vocab size: 1860, max sequence length: 750
Using NVIDIA A100-PCIE-40GB for training
Set float32_matmul_precision to medium
-- Initializing scmaskgit model
/home/stuke1/perturbgen/Perturbgen/perturbgen/Model/trainer.py:673: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_masking_path, map_location='cpu')
Start datamodule
Using device gpu.
No protocol specified
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
loading, please wait...
loading, please wait...
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/3
Current working directory: /home/stuke1/perturbgen
Loading and preprocessing data...
Loading 3_10h_LPS.dataset...
Current working directory: /home/stuke1/perturbgen
Loading and preprocessing data...
Loading 3_10h_LPS.dataset...
Loading 2_6h_LPS.dataset...
Loading 2_6h_LPS.dataset...
Loading 1_90m_LPS.dataset...
Loading 1_90m_LPS.dataset...
Loading 3_10h_LPS.h5ad...
Loading 3_10h_LPS.h5ad...
[rank: 1] Seed set to 42
[rank: 2] Seed set to 42
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Loading 2_6h_LPS.h5ad...
Loading 2_6h_LPS.h5ad...
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Loading 1_90m_LPS.h5ad...
Loading 1_90m_LPS.h5ad...
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
---PerturbGen training --- 
Target vocab size: 1860, max sequence length: 750
---PerturbGen training --- 
Target vocab size: 1860, max sequence length: 750
Using NVIDIA A100-PCIE-40GB for training
Set float32_matmul_precision to medium
-- Initializing scmaskgit model
Using NVIDIA A100-PCIE-40GB for training
Set float32_matmul_precision to medium
-- Initializing scmaskgit model
/home/stuke1/perturbgen/Perturbgen/perturbgen/Model/trainer.py:673: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_masking_path, map_location='cpu')
/home/stuke1/perturbgen/Perturbgen/perturbgen/Model/trainer.py:673: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_masking_path, map_location='cpu')
Start datamodule
Using device gpu.
Start datamodule
Using device gpu.
No protocol specified
No protocol specified
Initializing distributed: GLOBAL_RANK: 2, MEMBER: 3/3
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/3
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 3 processes
----------------------------------------------------------------------------------------------------

wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id acdx9vut.
wandb: Tracking run with wandb version 0.17.9
wandb: W&B syncing is set to `offline` in this directory.  
wandb: Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [5,6,7]
LOCAL_RANK: 2 - CUDA_VISIBLE_DEVICES: [5,6,7]
LOCAL_RANK: 1 - CUDA_VISIBLE_DEVICES: [5,6,7]

  | Name             | Type             | Params | Mode 
--------------------------------------------------------------
0 | pretrained_model | PerturbGen       | 91.2 M | train
1 | decoder          | CountDecoder     | 95.5 M | train
2 | mse              | MeanSquaredError | 0      | train
--------------------------------------------------------------
4.3 M     Trainable params
91.2 M    Non-trainable params
...
wandb: WARNING Serializing object of type AnnData that is 263805505 bytes
wandb: WARNING Serializing object of type AnnData that is 278362328 bytes
wandb: WARNING Serializing object of type AnnData that is 279715640 bytes
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:105: Total length of `list` across ranks is zero. Please make sure this was your intention.
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
Epoch 0:   0%|          | 0/9874 [00:00<?, ?it/s] 
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/scvi/distributions/_negative_binomial.py:93: UserWarning: Specified kernel cache directory could not be created! This disables kernel caching. Specified directory is /home/stuke1/.cache/torch/kernels. This warning will appear only once per process. (Triggered internally at ../aten/src/ATen/native/cuda/jit_utils.cpp:1442.)
  + lgamma(x + theta)
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:78: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 5. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
Epoch 0: 100%|█████████▉| 9870/9874 [20:40<00:00,  7.96it/s, v_num=9vut, train/loss_step=1.95e+3, train/mse_step=13.90]
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:78: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 4. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:431: It is recommended to use `self.log('train/loss', ..., sync_dist=True)` when logging on epoch level in distributed setting to accumulate the metric across devices.
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:431: It is recommended to use `self.log('train/mse', ..., sync_dist=True)` when logging on epoch level in distributed setting to accumulate the metric across devices.
Epoch 4: 100%|██████████| 9874/9874 [22:54<00:00,  7.19it/s, v_num=9vut, train/loss_step=1.55e+3, train/mse_step=3.410, train/loss_epoch=1.41e+3, train/mse_epoch=3.670, train/emd=0.129]
Epoch 4, global step 49370: 'train/mse' reached 3.66602 (best 3.66602), saving model to '/home/stuke1/perturbgen/T_perturb/res/count/checkpoints/20260730_1020_cellgen_train_count_lr_0.001_wd_0.001_batch_16_drop_0.1_zinb_tp_1-2-3_s_42_pos_time_pos_sin_m_pow-epoch=04.ckpt' as top 1
Epoch 9: 100%|██████████| 9874/9874 [22:14<00:00,  7.40it/s, v_num=9vut, train/loss_step=1.29e+3, train/mse_step=0.952, train/loss_epoch=1.4e+3, train/mse_epoch=3.410, train/emd=0.122] 
Epoch 9, global step 98740: 'train/mse' reached 3.40852 (best 3.40852), saving model to '/home/stuke1/perturbgen/T_perturb/res/count/checkpoints/20260730_1020_cellgen_train_count_lr_0.001_wd_0.001_batch_16_drop_0.1_zinb_tp_1-2-3_s_42_pos_time_pos_sin_m_pow-epoch=09.ckpt' as top 2
Epoch 14: 100%|██████████| 9874/9874 [22:16<00:00,  7.39it/s, v_num=9vut, train/loss_step=1.34e+3, train/mse_step=1.140, train/loss_epoch=1.4e+3, train/mse_epoch=3.400, train/emd=0.120]
Epoch 14, global step 148110: 'train/mse' reached 3.39703 (best 3.39703), saving model to '/home/stuke1/perturbgen/T_perturb/res/count/checkpoints/20260730_1020_cellgen_train_count_lr_0.001_wd_0.001_batch_16_drop_0.1_zinb_tp_1-2-3_s_42_pos_time_pos_sin_m_pow-epoch=14.ckpt' as top 3
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/stuke1/perturbgen/Perturbgen/perturbgen/__main__.py", line 88, in <module>
    main()
  File "/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/click/core.py", line 1157, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/click/core.py", line 1078, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/click/core.py", line 1688, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/click/core.py", line 1434, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/click/core.py", line 783, in invoke
    return __callback(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/stuke1/perturbgen/Perturbgen/perturbgen/__main__.py", line 75, in train_decoder
    main(args)
  File "/home/stuke1/perturbgen/Perturbgen/perturbgen/train.py", line 676, in main
    trainer.fit(decoder_module, data_module)
...
wandb: You can sync this run to the cloud by running:
wandb: wandb sync logs/wandb/offline-run-20260730_102129-acdx9vut
wandb: Find logs at: logs/wandb/offline-run-20260730_102129-acdx9vut/logs
wandb: WARNING The new W&B backend becomes opt-out in version 0.18.0; try it out with `wandb.require("core")`! See https://wandb.me/wandb-core for more information.
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
---------------------------------------------------------------------------
CalledProcessError                        Traceback (most recent call last)
Cell In[5], line 9
      6 env.setdefault("WANDB_MODE", "offline")
      8 print("Launching count-decoder train on GPUs", env["CUDA_VISIBLE_DEVICES"])
----> 9 subprocess.run(count_cmd, check=True, cwd=str(WORKSPACE), env=env)

File ~/miniforge3/envs/perturbgen/lib/python3.11/subprocess.py:571, in run(input, capture_output, timeout, check, *popenargs, **kwargs)
    569     retcode = process.poll()
    570     if check and retcode:
--> 571         raise CalledProcessError(retcode, process.args,
    572                                  output=stdout, stderr=stderr)
    573 return CompletedProcess(process.args, retcode, stdout, stderr)

CalledProcessError: Command '['python', '-m', 'perturbgen', 'train-decoder', '--train_mode', 'count', '--split', 'False', '--splitting_mode', 'stratified', '--output_dir', '/home/stuke1/perturbgen/T_perturb/res/count', '--ckpt_masking_path', '/home/stuke1/perturbgen/T_perturb/res/masking/checkpoints/20260729_1751_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_0-epoch=19.ckpt', '--src_dataset', '/home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset', '--tgt_dataset_folder', '/home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt', '--src_adata', '/home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad', '--tgt_adata_folder', '/home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt', '--mapping_dict_path', '/home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl', '--batch_size', '16', '--epochs', '16', '--count_lr', '0.001', '--cellgen_lr', '0.0001', '--cellgen_wd', '0.0001', '--count_wd', '0.001', '--mlm_prob', '0.3', '--n_workers', '4', '--num_layers', '6', '--d_ff', '32', '--loss_mode', 'zinb', '--pred_tps', '1', '2', '3', '--var_list', 'cell_type_harmonized', 'time_after_LPS', '--encoder', 'scmaskgit', '--count_dropout', '0.1', '--use_positional_encoding', 'False', '--layer_norm', 'True', '--context_mode', 'True', '--encoder_path', '/home/stuke1/perturbgen/Perturbgen/pretraining_cohort/20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_42-epoch=00.ckpt', '--pos_encoding_mode', 'time_pos_sin', '--mask_scheduler', 'pow', '--num_node', '1', '--d_model', '768', '--ckpt_every_n_epochs', '5', '--wandb_mode', 'offline']' returned non-zero exit status 1.
[rank1]:[E730 16:22:05.526956405 ProcessGroupNCCL.cpp:616] [Rank 1] Watchdog caught collective operation timeout: WorkNCCL(SeqNum=444450, OpType=ALLREDUCE, NumelIn=1, NumelOut=1, Timeout(ms)=1800000) ran for 1800041 milliseconds before timing out.
[rank1]:[E730 16:22:05.527200344 ProcessGroupNCCL.cpp:1785] [PG ID 0 PG GUID 0(default_pg) Rank 1] Exception (either an error or timeout) detected by watchdog at work: 444450, last enqueued NCCL work: 444450, last completed NCCL work: 444449.
[rank1]:[E730 16:22:05.527210714 ProcessGroupNCCL.cpp:1834] [PG ID 0 PG GUID 0(default_pg) Rank 1] Timeout at NCCL work: 444450, last enqueued NCCL work: 444450, last completed NCCL work: 444449.
[rank1]:[E730 16:22:05.527224114 ProcessGroupNCCL.cpp:630] [Rank 1] Some NCCL operations have failed or timed out. Due to the asynchronous nature of CUDA kernels, subsequent GPU operations might run on corrupted/incomplete data.
[rank1]:[E730 16:22:05.527227834 ProcessGroupNCCL.cpp:636] [Rank 1] To avoid data inconsistency, we are taking the entire process down.
[rank1]:[E730 16:22:05.528435077 ProcessGroupNCCL.cpp:1595] [PG ID 0 PG GUID 0(default_pg) Rank 1] Process group watchdog thread terminated with exception: [Rank 1] Watchdog caught collective operation timeout: WorkNCCL(SeqNum=444450, OpType=ALLREDUCE, NumelIn=1, NumelOut=1, Timeout(ms)=1800000) ran for 1800041 milliseconds before timing out.
Exception raised from checkTimeout at ../torch/csrc/distributed/c10d/ProcessGroupNCCL.cpp:618 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::string) + 0x96 (0x7ffa00881446 in /home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/torch/lib/libc10.so)
frame #1: c10d::ProcessGroupNCCL::WorkNCCL::checkTimeout(std::optional<std::chrono::duration<long, std::ratio<1l, 1000l> > >) + 0x282 (0x7ffa01b94772 in /home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/torch/lib/libtorch_cuda.so)
frame #2: c10d::ProcessGroupNCCL::watchdogHandler() + 0x233 (0x7ffa01b9bbb3 in /home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/torch/lib/libtorch_cuda.so)
frame #3: c10d::ProcessGroupNCCL::ncclCommWatchdog() + 0x14d (0x7ffa01b9d61d in /home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/torch/lib/libtorch_cuda.so)
frame #4: <unknown function> + 0x145c0 (0x7ffa4a51e5c0 in /home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/torch/lib/libtorch.so)
frame #5: <unknown function> + 0x8609 (0x7ffa4dbf9609 in /lib/x86_64-linux-gnu/libpthread.so.0)
frame #6: clone + 0x43 (0x7ffa4d9c4353 in /lib/x86_64-linux-gnu/libc.so.6)

terminate called after throwing an instance of 'c10::DistBackendError'
  what():  [PG ID 0 PG GUID 0(default_pg) Rank 1] Process group watchdog thread terminated with exception: [Rank 1] Watchdog caught collective operation timeout: WorkNCCL(SeqNum=444450, OpType=ALLREDUCE, NumelIn=1, NumelOut=1, Timeout(ms)=1800000) ran for 1800041 milliseconds before timing out.
Exception raised from checkTimeout at ../torch/csrc/distributed/c10d/ProcessGroupNCCL.cpp:618 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::string) + 0x96 (0x7ffa00881446 in /home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/torch/lib/libc10.so)
frame #1: c10d::ProcessGroupNCCL::WorkNCCL::checkTimeout(std::optional<std::chrono::duration<long, std::ratio<1l, 1000l> > >) + 0x282 (0x7ffa01b94772 in /home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/torch/lib/libtorch_cuda.so)
frame #2: c10d::ProcessGroupNCCL::watchdogHandler() + 0x233 (0x7ffa01b9bbb3 in /home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/torch/lib/libtorch_cuda.so)
frame #3: c10d::ProcessGroupNCCL::ncclCommWatchdog() + 0x14d (0x7ffa01b9d61d in /home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/torch/lib/libtorch_cuda.so)
frame #4: <unknown function> + 0x145c0 (0x7ffa4a51e5c0 in /home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/torch/lib/libtorch.so)
frame #5: <unknown function> + 0x8609 (0x7ffa4dbf9609 in /lib/x86_64-linux-gnu/libpthread.so.0)
frame #6: clone + 0x43 (0x7ffa4d9c4353 in /lib/x86_64-linux-gnu/libc.so.6)
...
[octavian:92422] [ 8] /home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/torch/lib/libtorch.so(+0x145c0)[0x7f9e7f3895c0]
[octavian:92422] [ 9] /lib/x86_64-linux-gnu/libpthread.so.0(+0x8609)[0x7f9e82a64609]
[octavian:92422] [10] /lib/x86_64-linux-gnu/libc.so.6(clone+0x43)[0x7f9e8282f353]
[octavian:92422] *** End of error message ***
env.setdefault("WANDB_MODE", "offline")

print("Launching count-decoder train on GPUs", env["CUDA_VISIBLE_DEVICES"])
subprocess.run(count_cmd, check=True, cwd=str(WORKSPACE), env=env)

Launching count-decoder train on GPUs 5,6,7
loading, please wait...
Current working directory: /home/stuke1/perturbgen
Loading and preprocessing data...
Loading 3_10h_LPS.dataset...
Loading 2_6h_LPS.dataset...
Loading 1_90m_LPS.dataset...
Loading 3_10h_LPS.h5ad...


Seed set to 42
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading 2_6h_LPS.h5ad...


/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading 1_90m_LPS.h5ad...


...

Epoch 9, global step 98740: 'train/mse' reached 3.40852 (best 3.40852), saving model to '/home/stuke1/perturbgen/T_perturb/res/count/checkpoints/20260730_1020_cellgen_train_count_lr_0.001_wd_0.001_batch_16_drop_0.1_zinb_tp_1-2-3_s_42_pos_time_pos_sin_m_pow-epoch=09.ckpt' as top 2


Epoch 14: 100%|██████████| 9874/9874 [22:16<00:00,  7.39it/s, v_num=9vut, train/loss_step=1.34e+3, train/mse_step=1.140, train/loss_epoch=1.4e+3, train/mse_epoch=3.400, train/emd=0.120]

Epoch 14, global step 148110: 'train/mse' reached 3.39703 (best 3.39703), saving model to '/home/stuke1/perturbgen/T_perturb/res/count/checkpoints/20260730_1020_cellgen_train_count_lr_0.001_wd_0.001_batch_16_drop_0.1_zinb_tp_1-2-3_s_42_pos_time_pos_sin_m_pow-epoch=14.ckpt' as top 3
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/stuke1/perturbgen/Perturbgen/perturbgen/__main__.py", line 88, in <module>
    main()
  File "/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/click/core.py", line 1157, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/click/core.py", line 1078, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/click/core.py", line 1688, in invoke
    return _process_result(sub_ctx.co

CalledProcessError: Command '['python', '-m', 'perturbgen', 'train-decoder', '--train_mode', 'count', '--split', 'False', '--splitting_mode', 'stratified', '--output_dir', '/home/stuke1/perturbgen/T_perturb/res/count', '--ckpt_masking_path', '/home/stuke1/perturbgen/T_perturb/res/masking/checkpoints/20260729_1751_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_0-epoch=19.ckpt', '--src_dataset', '/home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset', '--tgt_dataset_folder', '/home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt', '--src_adata', '/home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad', '--tgt_adata_folder', '/home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt', '--mapping_dict_path', '/home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.

[rank1]:[E730 16:22:05.526956405 ProcessGroupNCCL.cpp:616] [Rank 1] Watchdog caught collective operation timeout: WorkNCCL(SeqNum=444450, OpType=ALLREDUCE, NumelIn=1, NumelOut=1, Timeout(ms)=1800000) ran for 1800041 milliseconds before timing out.
[rank1]:[E730 16:22:05.527200344 ProcessGroupNCCL.cpp:1785] [PG ID 0 PG GUID 0(default_pg) Rank 1] Exception (either an error or timeout) detected by watchdog at work: 444450, last enqueued NCCL work: 444450, last completed NCCL work: 444449.
[rank1]:[E730 16:22:05.527210714 ProcessGroupNCCL.cpp:1834] [PG ID 0 PG GUID 0(default_pg) Rank 1] Timeout at NCCL work: 444450, last enqueued NCCL work: 444450, last completed NCCL work: 444449.
[rank1]:[E730 16:22:05.527224114 ProcessGroupNCCL.cpp:630] [Rank 1] Some NCCL operations have failed or timed out. Due to the asynchronous nature of CUDA kernels, subsequent GPU operations might run on corrupted/incomplete data.
[rank1]:[E730 16:22:05.527227834 ProcessGroupNCCL.cpp:636] [Rank 1] To avoid data in